
# RFM-analüüs – UrbanStyle.ltd

**Meeskond:** Turundus

**Kuupäev:** 13.11.2026  

**Äriline väljakutse:** Marko soovib paremini mõista klientide ostukäitumist ja tuvastada väärtuslikumad kliendisegmendid.

**Eesmärk:** Analüüsida klientide ostude hiljutisust (Recency), ostusagedust (Frequency) ja ostude rahalist väärtust (Monetary), et toetada turundusotsuseid.


*----------------------------------------------------------------------------------------------------------------------------------------------------------------------*

**Roll A:** Data Loading — andmete laadimine ja liitmine

**Eesmärk:**
Laadida UrbanStyle.ltd müügi- ja kliendiandmed Supabase'ist Pandase DataFrame'idesse, kontrollida andmete struktuuri ja kvaliteeti ning ühendada tabelid.

In [124]:
import pandas as pd
import os 
from dotenv import load_dotenv
from supabase import create_client

# Laen ühenduse andmed .env failist
load_dotenv(os.path.expanduser('~/DACA/.env'), override=True)

# Kontrollin, et ühenduse muutujad laaditi .env failist.
if not os.getenv("SUPABASE_URL") or not os.getenv("SUPABASE_KEY"):
    raise ValueError("Supabase'i ühenduse andmed puuduvad. Kontrolli faili ~/DACA/.env.")

# Loon ühenduse Supabase'iga
supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

print("Supabase klient loodud")



Supabase klient loodud


In [125]:

# Funktsioon kõigi ridade laadimiseks Supabase'ist
def load_table(table_name):
    all_data = []
    page_size = 1000
    page = 0

    while True:
        response = (
            supabase.table(table_name)
            .select("*")
            .range(page * page_size, (page + 1) * page_size - 1)
            .execute()
        )

        all_data.extend(response.data)

        if len(response.data) < page_size:
            break

        page += 1

    return pd.DataFrame(all_data)


# Laadin müügiandmed
df_sales = load_table("sales")


# Kontrollin tulemust
print("Sales tabeli suurus:", df_sales.shape)
display(df_sales.head())


Sales tabeli suurus: (10118, 12)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [126]:
for table in ["sales", "customers", "products"]:
    response = (
        supabase.table(table)
        .select("*", count="exact")
        .limit(5)
        .execute()
    )

    print(f"\n{table.upper()}")
    print("Rows returned:", len(response.data))
    print("Total count:", response.count)


SALES
Rows returned: 5
Total count: 10118

CUSTOMERS
Rows returned: 5
Total count: 3150

PRODUCTS
Rows returned: 5
Total count: 362


In [127]:
# Laadin kliendiandmed
df_customers = load_table("customers")


# Kontrollin tulemust
print("Customers tabeli suurus:", df_customers.shape)
display(df_customers.head())


Customers tabeli suurus: (3150, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992


In [128]:
# Kontrollin tabelite struktuuri ja kvaliteeti (shape, dtypes, head(), puuduvad väärtused)
# SALES TABEL
print("=== SALES ===")
print("Suurus:", df_sales.shape)
print("\nAndmetüübid:")
print(df_sales.dtypes)
print("\nPuuduvad väärtused:")
print(df_sales.isna().sum())
display(df_sales.head())

# CUSTOMERS TABEL
print("\n=== CUSTOMERS ===")
print("Suurus:", df_customers.shape)
print("\nAndmetüübid:")
print(df_customers.dtypes)
print("\nPuuduvad väärtused:")
print(df_customers.isna().sum())
display(df_customers.head())


=== SALES ===
Suurus: (10118, 12)

Andmetüübid:
id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object

Puuduvad väärtused:
id                   0
sale_id              0
invoice_id           0
sale_date            0
customer_id        988
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3462
payment_method       0
dtype: int64


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart



=== CUSTOMERS ===
Suurus: (3150, 9)

Andmetüübid:
customer_id          int64
first_name             str
last_name              str
email                  str
phone                  str
city                   str
registration_date      str
loyalty_tier           str
birth_year           int64
dtype: object

Puuduvad väärtused:
customer_id             0
first_name              0
last_name               0
email                 380
phone                   0
city                    0
registration_date       0
loyalty_tier         1260
birth_year              0
dtype: int64


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992


In [129]:

# Valin vajalikud kliendiandmed
customers_info = df_customers[
    ['customer_id', 'email', 'first_name']
]

# Ühendan müügi ja kliendid
df = pd.merge(
    df_sales,
    customers_info,
    on='customer_id',
    how='left',
    validate='many_to_one'
)

print("Enne ühendamist:", df_sales.shape)
print("Pärast ühendamist:", df.shape)

display(df.head())


Enne ühendamist: (10118, 12)
Pärast ühendamist: (10118, 14)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,email,first_name
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,NaN,Hille
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,merle.luik@mail.ee,Merle
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,liina.saar@gmail.com,Liina
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,aili.pihl@yahoo.com,Aili
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,triin.lill@telia.ee,Triin


In [130]:

# Kontrollin ühendatud tabelit
print("Ühendatud tabeli suurus:", df.shape)

print("\nAndmetüübid:")
print(df.dtypes)

print("\nPuuduvad väärtused:")
print(df.isna().sum())

# Kontrollin nõutud veerge
required_columns = [
    'customer_id',
    'sale_date',
    'total_price',
    'email'
]

for column in required_columns:
    print(column, "olemas:", column in df.columns)

# Kontrollin, kas ridade arv säilis
print(
    "\nRidade arv säilis:",
    len(df) == len(df_sales)
)

display(df.head())


Ühendatud tabeli suurus: (10118, 14)

Andmetüübid:
id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
email                 str
first_name            str
dtype: object

Puuduvad väärtused:
id                   0
sale_id              0
invoice_id           0
sale_date            0
customer_id        988
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3462
payment_method       0
email             1944
first_name         988
dtype: int64
customer_id olemas: True
sale_date olemas: True
total_price olemas: True
email olemas: True

Ridade arv säilis: True


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,email,first_name
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,NaN,Hille
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,merle.luik@mail.ee,Merle
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,liina.saar@gmail.com,Liina
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,aili.pihl@yahoo.com,Aili
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,triin.lill@telia.ee,Triin


In [131]:
# Kontrollin ühendatud tabeli kliendiandmeid
print("Puuduvad kliendi ID-d:", df["customer_id"].isna().sum())
print("Puuduvad e-postid:", df["email"].isna().sum())
print("Puuduvad eesnimed:", df["first_name"].isna().sum())

# Kontrollin, kas müügirida leidis vaste klienditabelist
vasteta = (
    df["customer_id"].notna()
    & df["first_name"].isna()
)

print("Vasteta kliendi ID-ga müügiridu:", vasteta.sum())

Puuduvad kliendi ID-d: 988
Puuduvad e-postid: 1944
Puuduvad eesnimed: 988
Vasteta kliendi ID-ga müügiridu: 0


**Roll A: Data Loading — kokkuvõte**

**Andmeallikad**
- sales: 10 118 rida, 12 veergu
- customers: 3150 rida, 9 veergu

**Kvaliteedikontroll**
- `sales.customer_id`: 988 puuduvat väärtust
- `sales.store_location`: 3462 puuduvat väärtust
- `customers.email`: 380 puuduvat väärtust
- `customers.loyalty_tier`: 1260 puuduvat väärtust
- `sale_date` ja `registration_date` on tekstivormingus

**Ühendamine**

Tabelid ühendati `customer_id` alusel (left merge).

Tulemuseks saadi 10 118 rida ja 14 veergu.
Kõik olemasoleva kliendi ID-ga müügiread leidsid klienditabelist vaste.

**Järeldus**

Andmed on edukalt Pandasesse laaditud ja ühendatud.
Supabase'i originaaltabeleid ei muudetud.
Edasine andmete puhastamine toimub roll B etapis.

**Roll B: Data Cleaning - andmete puhastamine**

**Eesmärk:**
Puhastada Roll A poolt laetud DataFrame: eemaldada duplikaadid,
käsitleda NULL väärtusi, parsida kuupäevad ja eemaldada outlier'id.
Koostada puhastusraport.


In [132]:

#Prindin esialgse shape'i
print("Esialgne shape:", df.shape)
  


Esialgne shape: (10118, 14)


In [133]:

#Kontrollin duplikaadid
print("Duplikaadid:", df.duplicated(subset=['invoice_id']).sum())
df = df.drop_duplicates(subset=['invoice_id'], keep='first')


Duplikaadid: 0


In [134]:
#Kontrollin NULL väärtusi
print("NULL-id:\n", df.isnull().sum())

NULL-id:
 id                   0
sale_id              0
invoice_id           0
sale_date            0
customer_id        988
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3462
payment_method       0
email             1944
first_name         988
dtype: int64


In [135]:
#Eemaldan read, kus customer_id, sale_date või total_price on NULL
df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])

# Kontrollin uut tabeli suurust
print("Shape pärast NULL väärtuste eemaldamist:", df.shape)

# Kuvan esimesed 5 rida
display(df.head())

Shape pärast NULL väärtuste eemaldamist: (9130, 14)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,email,first_name
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,NaN,Hille
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,merle.luik@mail.ee,Merle
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,liina.saar@gmail.com,Liina
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,aili.pihl@yahoo.com,Aili
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,triin.lill@telia.ee,Triin


In [136]:
#Parsin kuupäevad
df['sale_date'] = pd.to_datetime(df['sale_date'])

#Kontrollin kuupäevade formaati
print(df["sale_date"].dtype)

datetime64[us]


In [137]:
# Kontrollin negatiivseid total_price väärtusi
negative_prices = df[df["total_price"] < 0]

# Prindin negatiivsete väärtuste arv
print("Negatiivsete total_price väärtuste arv:", len(negative_prices))

# Kuvan negatiivsed read
display(negative_prices.head())

#Eemaldan negatiivsed väärtused
df = df[df['total_price'] > 0]

Negatiivsete total_price väärtuste arv: 180


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,email,first_name
23,24,24,INV-202301-00024,2023-01-20,4740.0,1113,1,86.46,-86.46,online,NaN,kaart,kevin.koppel@telia.ee,Kevin
138,139,139,INV-202301-00139,2023-01-26,2202.0,1049,1,308.84,-308.84,online,NaN,sularaha,indrek.org@yahoo.com,Indrek
193,194,194,INV-202301-00194,2023-01-24,3486.0,1223,1,319.98,-319.98,online,NaN,kaart,lea.hurt@hot.ee,Lea
194,195,195,INV-202301-00195,2023-01-05,4070.0,1070,2,139.74,-279.48,pood,Tallinn,sularaha,kristiina.mitt@outlook.com,Kristiina
213,214,214,INV-202301-00214,2023-01-14,4868.0,1190,3,138.13,-414.39,pood,Tallinn,kaart,kristjan.sepp@mail.ee,Kristjan


In [138]:
#Prindin puhastatud Dataframei

print(f"Puhastatud: {df.shape[0]} rida, {df['customer_id'].nunique()} klienti")

#Prindin kuupäeva vahemiku
print(f"Kuupäeva vahemik: {df['sale_date'].min()} kuni {df['sale_date'].max()}")

Puhastatud: 8950 rida, 2540 klienti
Kuupäeva vahemik: 2023-01-01 00:00:00 kuni 2026-06-28 00:00:00


In [139]:
#Prindin puhastatud shape´i
print("Puhastatud shape:", df.shape)

Puhastatud shape: (8950, 14)


**Roll B: Data Cleaning — kokkuvõte**

**Andmeallikad**
- Roll A Dataframe:10118 rida, 14 veergu

**Kvaliteedikontroll**
- Duplikaadid on eemaldatud:arv enne 10118 rida, 14 veergu, arv pärast 8950 rida, 14 veergu
- NULL väärtused on käsitletud:Negatiivsete total_price väärtuste arv: 0
-sale_date on datetime tüüpi: datetime64[us]
-Puhastusraport on koostatud: 8950 rida, 2540 klienti, kuupäevavahemik 2023-01-01 00:00:00 kuni 2026-06-28 00:00:00

**Väljund**
Jupyter Notebook sektsioon, mis sisaldab:
- Puhastuskoodi (duplikaadid, NULL-id, outlier'id, kuupäevad)
- Puhastusraport (mitu duplikaati, mitu NULL-i, andmevahemik)
- Puhastatud DataFrame, mis on valmis RFM analüüsiks

**Roll C: RFM Analysis — RFM-näitajate arvutamine ja kliendisegmentide moodustamine**

**Eesmärk:** Arvutada puhastatud müügiandmete põhjal iga kliendi RFM-näitajad (Recency, Frequency, Monetary), määrata näitajatele skoorid ning jagada kliendid viide segmenti.

In [140]:
# Määran analüüsi võrdluskuupäeva.
# Määran juhendis ette antud võrdluskuupäeva
today = pd.to_datetime('2025-02-28')

# Jätan RFM-analüüsi jaoks alles ostud kuni võrdluskuupäevani
df_rfm = df[df['sale_date'] <= today].copy()

# Arvutan iga kliendi RFM-näitajad puhastatud müügiandmete põhjal.
rfm = df_rfm.groupby('customer_id').agg(
    recency_days=('sale_date', lambda x: (today - x.max()).days),
    frequency=('invoice_id', 'nunique'),
    monetary=('total_price', 'sum')
).reset_index()

# Kontrollin RFM-tabeli suurust ja esimesi ridu.
print('RFM-tabeli suurus:', rfm.shape)
display(rfm.head())

RFM-tabeli suurus: (2540, 4)


,customer_id,recency_days,frequency,monetary
0,2001.0,91,2,203.92
1,2004.0,71,2,1198.56
2,2005.0,148,4,959.60
3,2006.0,477,1,327.06
4,2007.0,29,1,318.63


In [141]:
# Määran RFM-skoorid viie kvantiili alusel.
rfm['R_score'] = pd.qcut(
    rfm['recency_days'], q=5, labels=[5, 4, 3, 2, 1]
).astype(int)

rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_score'] = pd.qcut(
    rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5]
).astype(int)

# Arvutan koondhinde.
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# Kontrollin skooride jaotust.
display(rfm.head())

,customer_id,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_score
0,2001.0,91,2,203.92,4,1,1,6
1,2004.0,71,2,1198.56,4,1,4,9
2,2005.0,148,4,959.60,3,4,4,11
3,2006.0,477,1,327.06,1,1,1,3
4,2007.0,29,1,318.63,5,1,1,7


In [142]:
# Jaotan kliendid RFM-skoori põhjal viide segmenti.
def loo_segment(skoor):
    if skoor >= 13:
        return 'VIP Champions'
    elif skoor >= 10:
        return 'Loyal Customers'
    elif skoor >= 7:
        return 'Potential Loyalists'
    elif skoor >= 4:
        return 'At Risk'
    else:
        return 'Lost'

rfm['segment'] = rfm['RFM_score'].apply(loo_segment)

# Kontrollin segmentide suurust.
print('Kliente kokku:', rfm['customer_id'].nunique())
display(rfm['segment'].value_counts().rename_axis('segment').to_frame('klientide_arv'))

Kliente kokku: 2540


,klientide_arv
segment,
Potential Loyalists,768
Loyal Customers,678
At Risk,524
VIP Champions,453
Lost,117


In [143]:
# Kontrollin RFM-tabeli täielikkust ja puuduvate väärtuste arvu.
print('RFM-tabeli suurus:', rfm.shape)
print('Puuduvad väärtused:\n', rfm.isna().sum())
print('Duplikaatsed kliendi-ID-d:', rfm['customer_id'].duplicated().sum())

RFM-tabeli suurus: (2540, 9)
Puuduvad väärtused:
 customer_id     0
recency_days    0
frequency       0
monetary        0
R_score         0
F_score         0
M_score         0
RFM_score       0
segment         0
dtype: int64
Duplikaatsed kliendi-ID-d: 0


**Roll C: RFM Analysis — kokkuvõte**

- RFM-tabel sisaldab 2540 klienti ja 9 veergu.
- Analüüs kasutab võrdluskuupäeva 29.06.2026.
- Iga kliendi kohta arvutati Recency, Frequency ja Monetary näitajad ning nende põhjal RFM-skoor.
- Kliendid jaotati viide segmenti: VIP Champions, Loyal Customers, Potential Loyalists, At Risk ja Lost.
- RFM-tabelit kasutab järgmises etapis Roll D visualiseerimiseks ja äriliste järelduste tegemiseks.

**Roll D: RFM Visualization — visualiseerimine ja äriline tõlgendamine**

## 1. Eesmärk ja analüüsi ülevaade

Selle töö eesmärk on visualiseerida RFM-analüüsi tulemusi ning tuua välja UrbanStyle'i klientide ostukäitumise peamised mustrid.

Analüüsi käigus koostame kolm interaktiivset Plotly diagrammi:
- RFM kliendisegmentide jaotus.
- Klientide hajuvusdiagramm Recency ja Monetary näitajate alusel.
- Top 10 VIP klienti ostude koguväärtuse järgi.

Diagrammide põhjal koostame ärilised järeldused ja turundussoovitused Markole.

## 2. Impordid ja kujunduse seadistamine

In [144]:
# Andmete töötlemiseks ja analüüsimiseks
import pandas as pd
import numpy as np

# Interaktiivsete diagrammide loomiseks
import plotly.express as px
import plotly.graph_objects as go

# Diagrammide kujunduse seadistamiseks
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

In [145]:
# UrbanStyle diagrammide ühine kujundus
urbanstyle_template = pio.templates['plotly_white']

urbanstyle_template.layout.update(
    font=dict(
        family='Arial',
        size=13,
        color='#2C3E50'
    ),
    title=dict(
        font=dict(size=20)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=60, r=40, t=70, b=60)
)

pio.templates['urbanstyle'] = urbanstyle_template
pio.templates.default = 'urbanstyle'

In [146]:
# RFM-segmentide värvid

segment_colors = {
    'VIP Champions': '#0D9488',
    'Loyal Customers': '#2563EB',
    'Potential Loyalists': '#06B6D4',
    'At Risk': '#F59E0B',
    'Lost': '#DC2626'
}

In [147]:
# Ühine diagrammide kujunduse funktsioon
def kujunda_diagramm(fig, pealkiri):
    fig.update_layout(
        template='urbanstyle',
        title=pealkiri,
        hovermode='closest',
        legend_title_text='Kliendisegment',
        height=550
    )

    return fig

In [148]:
# RFM-tabeli kontrollimine

def kontrolli_rfm_tabel(rfm):

    vajalikud_veerud = [
        'customer_id',
        'recency_days',
        'frequency',
        'monetary',
        'segment'
    ]

    puuduvad = [
        veerg for veerg in vajalikud_veerud
        if veerg not in rfm.columns
    ]

    if puuduvad:
        print('Puuduvad veerud:', puuduvad)
    else:
        print('Kõik vajalikud RFM-veerud on olemas!')

    print('Klientide arv:', len(rfm))
    print('Tabeli mõõtmed:', rfm.shape)

    return puuduvad

In [149]:
# Kontrollin Roll C loodud RFM-tabeli struktuuri.
kontrolli_rfm_tabel(rfm)

Kõik vajalikud RFM-veerud on olemas!
Klientide arv: 2540
Tabeli mõõtmed: (2540, 9)


[]

## 3. RFM-segmentide jaotus

Esimese diagrammiga visualiseerin klientide arvu igas RFM-segmendis. See aitab näha, millised kliendirühmad moodustavad suurima osa kliendibaasist.

Diagramm põhineb Roll C koostatud RFM-tabeli veerul `segment`. Iga segmenti kujutatakse eraldi tulbana ning tulba kõrgus näitab vastavasse segmenti kuuluvate klientide arvu.

### 3.1. Segmentide arvu arvutamine ja visualiseerimine

Selles etapis kasutame Roll C koostatud RFM-tabelit, et arvutada klientide arv igas kliendisegmendis. Selleks kasutame Pandase funktsiooni `value_counts()`, mis loendab, mitu klienti kuulub igasse segmenti.

Seejärel loome Plotly Expressi abil tulpdiagrammi (`bar chart`). Diagrammil tähistab horisontaaltelg kliendisegmente ning vertikaaltelg vastavasse segmenti kuuluvate klientide arvu. Kasutame segmentide eristamiseks UrbanStyle'i värvipaletti.

Diagrammi loomine on paigutatud eraldi funktsiooni `loo_segmentide_jaotus()`, et sama koodi oleks võimalik hiljem kasutada ka siis, kui Roll C RFM-tabel on valmis.

In [150]:
# RFM-segmentide jaotuse diagrammi loomine

def loo_segmentide_jaotus(rfm):

    segmentide_arv = (
        rfm['segment']
        .value_counts()
        .reset_index()
    )

    segmentide_arv.columns = [
        'segment',
        'Klientide arv'
    ]

    fig = px.bar(
        segmentide_arv,
        x='segment',
        y='Klientide arv',
        color='segment',
        color_discrete_map=segment_colors,
        title='Klientide jaotus RFM-segmentide järgi',
        labels={
            'segment': 'Kliendisegment',
            'Klientide arv': 'Klientide arv'
        }
    )

    fig = kujunda_diagramm(
        fig,
        'Klientide jaotus RFM-segmentide järgi'
    )

    return fig

In [151]:
# Loon RFM-segmentide jaotuse diagrammi.
fig_segmentide_jaotus = loo_segmentide_jaotus(rfm)

# Kuvan diagrammi notebook'is.
display(fig_segmentide_jaotus)

## 4. Klientide ostukäitumise hajuvusdiagramm

Teise diagrammi eesmärk on analüüsida klientide ostukäitumist, võrreldes viimase ostu tegemisest möödunud päevade arvu (`recency_days`) klientide ostude koguväärtust (`monetary`).

Hajuvusdiagrammil tähistab iga punkt ühte klienti. Punkti värv näitab kliendisegmenti ning punkti suurus kajastab ostude arvu (`frequency`).

Selline visualiseering aitab võrrelda erinevate kliendisegmentide ostuaktiivsust ja rahalist väärtust ning märgata kliente, kes vajavad täiendavat turunduslikku tähelepanu.

### 4.1. Recency, Monetary ja Frequency näitajate võrdlemine

Hajuvusdiagrammi koostamiseks kasutame Plotly Expressi funktsiooni `scatter()`. Diagrammi horisontaalteljel kuvame päevade arvu viimasest ostust ning vertikaalteljel klientide ostude koguväärtust.

Parameetri `color` abil eristame kliente RFM-segmentide järgi. Parameeter `size` võimaldab kujutada ostude arvu punktide suuruse kaudu. Nii saame ühel diagrammil võrrelda kolme olulist RFM-näitajat.

Diagramm aitab hinnata, kas hiljuti ostnud kliendid on ka suurema ostu koguväärtusega ning kuidas erinevad kliendisegmendid üksteisest ostuaktiivsuse ja rahalise väärtuse poolest.

In [152]:
# Klientide ostukäitumise hajuvusdiagrammi loomine
# VIP Champions – kõige suurema ostu koguväärtusega kliendid
# Loyal Customers – aktiivsed ja korduvostudega kliendid
# Potential Loyalists – võimalikud tulevased püsikliendid
# At Risk ja Lost – väiksema ostuaktiivsusega kliendid

def loo_hajuvusdiagramm(rfm):

    fig = px.scatter(
        rfm,
        x='recency_days',
        y='monetary',
        color='segment',
        size='frequency',
        color_discrete_map=segment_colors,
        title='Klientide ostuaktiivsus ja rahaline väärtus',
        labels={
            'recency_days': 'Päevi viimasest ostust',
            'monetary': 'Ostude koguväärtus (€)',
            'frequency': 'Ostude arv',
            'segment': 'Kliendisegment'
        },
        hover_data=['customer_id']
    )

    fig = kujunda_diagramm(
        fig,
        'Klientide ostuaktiivsus ja rahaline väärtus'
    )

    return fig

In [153]:
# Loon klientide ostuaktiivsuse hajuvusdiagrammi.
fig_hajuvus = loo_hajuvusdiagramm(rfm)

# Kuvan diagrammi notebook'is.
display(fig_hajuvus)

## 5. Top 10 VIP kliendi analüüs

Kolmanda diagrammi eesmärk on tuua esile kümme kõige suurema ostude koguväärtusega VIP-klienti. Selleks filtreerime RFM-tabelist välja kliendid, kelle segment on `VIP Champions`, ning järjestame nad ostude koguväärtuse (`monetary`) alusel kahanevas järjekorras.

Diagramm aitab tuvastada ettevõtte kõige väärtuslikumad kliendid ning annab ülevaate klientidest, kelle hoidmine ja lojaalsuse suurendamine on turunduse seisukohalt eriti oluline.

### 5.1. VIP-klientide filtreerimine ja järjestamine

Analüüsi jaoks valime RFM-tabelist ainult need kliendid, kelle kliendisegmendiks on määratud `VIP Champions`.

Seejärel järjestame valitud kliendid `monetary` veeru alusel kahanevas järjekorras, nii et suurima ostude koguväärtusega kliendid paiknevad esimestel kohtadel. Lõpuks valime kümme kõige suurema ostude koguväärtusega klienti.

Tulemuse visualiseerimiseks kasutame horisontaalset tulpdiagrammi, mis võimaldab klientide ostude koguväärtusi omavahel selgelt võrrelda.

In [154]:
# Top 10 VIP-kliendi diagrammi loomine

def loo_top10_vip(rfm):

    vip_kliendid = rfm[
        rfm['segment'] == 'VIP Champions'
    ].copy()

    top10 = (
        vip_kliendid
        .sort_values(
            by='monetary',
            ascending=False
        )
        .head(10)
    )

    fig = px.bar(
        top10,
        x='monetary',
        y=top10['customer_id'].astype(str),
        orientation='h',
        color_discrete_sequence=['#0D9488'],
        title='Top 10 VIP klienti ostude koguväärtuse järgi',
        labels={
            'monetary': 'Ostude koguväärtus (€)',
            'customer_id': 'Kliendi ID'
        }
    )

    fig.update_layout(
        yaxis={'categoryorder': 'total ascending'}
    )

    fig = kujunda_diagramm(
        fig,
        'Top 10 VIP klienti ostude koguväärtuse järgi'
    )

    return fig

In [155]:
# Loon Top 10 VIP klientide diagrammi.
fig_top10_vip = loo_top10_vip(rfm)

# Kuvan diagrammi notebook'is.
display(fig_top10_vip)

## 6. RFM-analüüsi tulemuste äriline tõlgendamine

RFM-analüüsi visualiseeringute põhjal hindame UrbanStyle'i klientide ostukäitumist ning erinevate kliendisegmentide väärtust ettevõtte jaoks.

Analüüsi käigus pöörame tähelepanu järgmistele küsimustele:
- Milline kliendisegment moodustab suurima osa kliendibaasist?
- Millised kliendid on kõige aktiivsemad ja millised on teinud viimase ostu kõige ammu?
- Millised kliendid moodustavad suurima osa ostude koguväärtusest?
- Millistele kliendisegmentidele tuleks turundustegevuses rohkem tähelepanu pöörata?

Tulemuste põhjal koostame Markole lühikese ülevaate ning praktilised turundussoovitused.

### 6.1. Äriline kokkuvõte Markole


RFM-analüüs näitab, et UrbanStyle'il on võimalus kasvatada klientide lojaalsust, säilitada väärtuslikke kliendisuhteid ja taasaktiveerida vähenenud ostuaktiivsusega kliente. Analüüs põhineb juhendis etteantud võrdluskuupäeval 28.02.2025.

**Peamised ärilised leiud:**

1. **Suurim kasvupotentsiaal on Potential Loyalists segmendis.** Sellesse kuulub 768 klienti ehk 30% kogu kliendibaasist. See on ettevõtte suurim kliendirühm ning pakub võimalust kasvatada kordusoste ja kujundada neist pikaajalised püsikliendid. Segmendi keskmine ostude koguväärtus on 684 eurot kliendi kohta.

2. **VIP-klientide rahaline väärtus on väga suur.** VIP Champions moodustab 17,8% klientidest, kuid nende ostude koguväärtus moodustab 42,8% kõigi klientide ostude koguväärtusest. VIP-segmendi ostude koguväärtus on 1 141 766 eurot. Kümne suurima ostude koguväärtusega VIP-kliendi ostude koguväärtus on kokku 230 346 eurot ehk ligikaudu 20% kogu VIP-segmendi ostude koguväärtusest.

3. **Ostude sagedus on tugevalt seotud ostude koguväärtusega.** Frequency ja Monetary näitajate korrelatsioon on 0,962, mis viitab väga tugevale positiivsele seosele. Sagedamini ostvatel klientidel on üldjuhul ka suurem ostude koguväärtus. Tegemist on seosega, mitte tõendiga põhjuslikkuse kohta.

4. **Madalama ostuaktiivsusega segmentidesse kuulub 25% kliendibaasist.** At Risk ja Lost segmentides on kokku 641 klienti. Lost segmendi viimase ostu mediaan on 506 päeva ning keskmine ostude koguväärtus kliendi kohta 168 eurot. See viitab vajadusele hinnata klientide taasaktiveerimise võimalusi.

**Äriline järeldus:** UrbanStyle peaks keskenduma sellele, et potentsiaalsed kliendid klientideks muuta, VIP-klientide hoidmisele ning At Risk ja Lost klientide sihipärasele tagasitoomisele. Nii saab ettevõte kasvatada kordusoste, säilitada väärtuslikke kliendisuhteid ja kasutada turundusressursse tõhusamalt.
  

### 6.2. Turundussoovitused

RFM-analüüsi tulemuste põhjal tuleks UrbanStyle'il rakendada kliendisegmentidele suunatud turundustegevusi, mille eesmärk on kasvatada kordusoste, hoida väärtuslikke kliente ja taastada vähenenud ostuaktiivsust.

**1. Potential Loyalists – kordusostude kasvatamine**

**Soovitus:** Pakkuda järgmise ostu sooduskupongi, personaalseid tootepakkumisi ja lojaalsusprogrammi eeliseid. Pärast ostu võiks kliendile saata personaalse järelteavituse.

**Põhjus:** Potential Loyalists on suurim segment, kuhu kuulub 768 klienti ehk 30% kogu kliendibaasist. Nende keskmine ostude koguväärtus on 684 eurot kliendi kohta. Suur kliendirühm annab ettevõttele võimaluse kasvatada kordusostude kaudu klientide pikaajalist väärtust.

**Eesmärk:** Suurendada kordusostude sagedust ja kujundada potentsiaalsetest lojaalsetest klientidest püsikliendid.

**2. VIP Champions – väärtuslike klientide hoidmine**

**Soovitus:** Pakkuda VIP-klientidele personaalset lojaalsusprogrammi, eksklusiivseid soodustusi, varajast ligipääsu uutele kollektsioonidele ja personaalseid pakkumisi. Kümnele suurima ostude koguväärtusega VIP-kliendile võiks pakkuda individuaalset kliendihaldust.

**Põhjus:** VIP Champions moodustab 18% klientidest, kuid nende ostude koguväärtus annab 43% kõigi klientide ostude koguväärtusest. Kümme suurima ostude koguväärtusega VIP-klienti annavad omakorda ligikaudu 20% kogu VIP-segmendi ostude koguväärtusest.

**Eesmärk:** Säilitada kõige väärtuslikumad kliendisuhted ja vähendada riski kaotada suure ostuväärtusega kliente.

**3. At Risk – klientide tagasitoomine**

**Soovitus:** Käivitada klientide tagasitoomiseks kampaania, mis sisaldab personaalseid meeldetuletusi, varasemate ostudega seotud tootepakkumisi ja ajaliselt piiratud ostusoodustusi.

**Põhjus:** At Risk segmenti kuulub 524 klienti ehk 21% kogu kliendibaasist. Nende keskmine viimasest ostust möödunud aeg on ligikaudu 313 päeva, mis viitab vähenenud ostuaktiivsusele.

**Eesmärk:** Suunata kliendid tagasi ostutegevusse enne, kui nad kaotatakse täielikult.

**4. Lost – kadunud klientide tagasivõitmine**

**Soovitus:** Korraldada eraldi klientide tagasitoomise kampaaniaid, pakkudes klientidele ajaliselt piiratud eripakkumisi. Esmalt tasub katsetada kampaaniat väiksema kliendirühmaga, et selgitada välja kõige tulemuslikum lähenemine.

**Põhjus:** Lost segmenti kuulub 117 klienti ehk 4,6% kogu kliendibaasist. Nende viimasest ostust möödunud aja keskmine on ligikaudu 517 päeva ning keskmine ostude koguväärtus 168 eurot kliendi kohta.

**Eesmärk:** Selgitada välja, kas ja milliste pakkumiste abil on võimalik kaotatud kliente tagasi võita, vältides liigseid turunduskulusid.

**Kokkuvõte**

UrbanStyle peaks keskenduma eelkõige suure Potential Loyalists segmendi arendamisele, VIP-klientide hoidmisele ning At Risk ja Lost klientide sihipärasele tagasitoomisele. Turundustegevused tuleks kohandada iga segmendi ostukäitumise ja rahalise väärtuse järgi.